In [0]:
%run ../config/config

In [0]:
# Instalación de dependencias 
%%capture
%pip install catboost optuna xgboost lightgbm category_encoders openpyxl fsspec

In [0]:
# Imports
import sys
import time
import pickle
import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.catboost
import mlflow.tracking._model_registry.utils
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"

from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
from mlflow.exceptions import MlflowException
from pyspark.sql import SparkSession

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger

In [0]:
# ==================================================
# Definición del modelo dummy (clase simple y detectable)
# ==================================================
class DummyModel:
    """
    Modelo dummy que devuelve probabilidad fija para pruebas.
    Serializable con pickle y compatible con MLflow (como sklearn).
    """

    def __init__(self, prob_class1=0.15):
        self.prob_class1 = prob_class1
        # Simula el atributo que algunos modelos reales tienen
        self.feature_names_in_ = None  # Se puede asignar si se conoce

    def predict_proba(self, X):
        """
        Devuelve matriz de probabilidades [P(clase0), P(clase1)].
        X puede ser numpy array o pandas DataFrame (se ignora).
        """
        n = len(X)
        prob_ones = np.full(n, self.prob_class1)
        prob_zeros = 1 - prob_ones
        return np.column_stack((prob_zeros, prob_ones))

    def predict(self, X):
        """Devuelve clase predicha (1 si probabilidad > 0.5)."""
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def create_dummy_model(prob_class1=0.15):
    """Factoría para crear una instancia del modelo dummy."""
    return DummyModel(prob_class1=prob_class1)

In [0]:
logger = MLOpsLogger(
    name='01_load_model',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '01_load_model'
    }
)

In [0]:
mlflow.set_registry_uri("databricks-uc")
logger.info(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

In [0]:
# ============================================================================
# 01_load_model.ipynb
# Propósito: Cargar modelo CatBoost desde pickle y registrarlo en Unity Catalog
#            con la firma (signature) requerida por UC. Incluye fallback a
#            FEATURE_NAMES desde config si el modelo no expone feature_names.
# ============================================================================
try:
    logger.log_stage_start('load_model', mes_vta=MES_VTA, environment=ENV, model_path=MODEL_FILE_PATH)
    start_time = time.time()

    logger.info("=" * 60)
    logger.info("CARGANDO MODELO DESDE PICKLE")
    logger.info("=" * 60)

    # ====================================================
    # 1. Cargar modelo real
    # ====================================================
    model = None
    is_dummy = False          # Inicializar como False (real)
    feature_names = None

    if os.path.exists(MODEL_FILE_PATH):
        file_size = os.path.getsize(MODEL_FILE_PATH)
        if file_size == 0:
            logger.warning(f"Archivo {MODEL_FILE_PATH} vacío. Se eliminará.")
            os.remove(MODEL_FILE_PATH)
        else:
            try:
                with open(MODEL_FILE_PATH, "rb") as file:
                    model = pickle.load(file)
                logger.info(f"✅ Modelo real cargado desde: {MODEL_FILE_PATH}")
                logger.info(f"Tipo: {type(model).__name__}")

                # Verificar si es una instancia del dummy
                is_dummy = isinstance(model, DummyModel)
                logger.info(f"¿Es modelo dummy? {is_dummy}")

                # Intentar extraer nombres de features (si existen)
                if hasattr(model, 'feature_names_') and model.feature_names_ is not None:
                    feature_names = model.feature_names_
                elif hasattr(model, 'feature_names_in_') and model.feature_names_in_ is not None:
                    feature_names = list(model.feature_names_in_)
                elif hasattr(model, '_feature_names') and model._feature_names is not None:
                    feature_names = model._feature_names
                else:
                    # Intento adicional para CatBoost (método get_feature_names)
                    try:
                        if hasattr(model, 'get_feature_names'):
                            feature_names = model.get_feature_names()
                    except:
                        pass

                if feature_names:
                    logger.info(f"Features extraídos del modelo: {len(feature_names)} (primeros 5: {feature_names[:5]})")
                else:
                    logger.warning("El modelo no expone nombres de features.")

            except (EOFError, pickle.UnpicklingError) as e:
                logger.error(f"Error en pickle: {e}. Archivo corrupto, eliminado.")
                os.remove(MODEL_FILE_PATH)
                model = None
                is_dummy = False
    else:
        logger.warning(f"No se encontró el archivo de modelo en {MODEL_FILE_PATH}")

    # ======================================================
    # 2. Si el modelo real no expone features, usar FEATURE_NAMES desde config
    # ======================================================
    if model is not None and not is_dummy and feature_names is None:
        if 'FEATURE_NAMES' in globals() and FEATURE_NAMES:
            feature_names = FEATURE_NAMES
            logger.info(f"Usando lista de features desde config.FEATURE_NAMES: {len(feature_names)} variables")
        else:
            logger.warning("No se encontró FEATURE_NAMES en config. No se podrá registrar en Unity Catalog.")

    # ====================================================
    # 3. Crear dummy si no hay modelo y está permitido
    # ====================================================
    if model is None and USE_DUMMY_MODEL:
        logger.warning("⚠️ Modelo real no encontrado. Creando modelo DUMMY.")
        model = create_dummy_model()
        is_dummy = True
        feature_names = None  # dummy no tiene features
        os.makedirs(os.path.dirname(MODEL_FILE_PATH), exist_ok=True)
        with open(MODEL_FILE_PATH, "wb") as f:
            pickle.dump(model, f)
        if os.path.getsize(MODEL_FILE_PATH) == 0:
            raise IOError("No se pudo guardar el modelo dummy")
        logger.info(f"Modelo dummy guardado en {MODEL_FILE_PATH}")

    if model is None:
        raise FileNotFoundError(f"No hay modelo en {MODEL_FILE_PATH} y USE_DUMMY_MODEL={USE_DUMMY_MODEL}")

    if not hasattr(model, 'predict_proba'):
        raise AttributeError("El modelo no tiene predict_proba")

    # ===========================================================
    # 4. Registrar en MLflow (solo si NO es dummy, hay nombre registrado y hay features)
    # ===========================================================
    registered_version = None
    run_id = None

    if not is_dummy and REGISTERED_MODEL_NAME and REGISTERED_MODEL_NAME.strip() and feature_names:
        logger.info("Registrando modelo real en Unity Catalog con firma...")
        try:
            import pandas as pd

            input_example = pd.DataFrame([[0.0] * len(feature_names)], columns=feature_names)
            proba_example = model.predict_proba(input_example)[:, 1]
            signature = infer_signature(input_example, proba_example)

            with mlflow.start_run(run_name=f"load_model_{MES_VTA}") as run:
                run_id = run.info.run_id

                # --- 1. Registrar el modelo ---
                mlflow.catboost.log_model(
                    cb_model=model,
                    artifact_path="model",
                    registered_model_name=REGISTERED_MODEL_NAME,
                    signature=signature,
                    input_example=input_example
                )
                logger.info(f"Modelo registrado | Run ID: {run_id} | Nombre: {REGISTERED_MODEL_NAME}")

                # --- 2. Obtener la versión registrada ---
                try:
                    client = MlflowClient()
                    # Usar el alias 'latest-databricks' que Databricks asigna automáticamente
                    latest_version = client.get_model_version_by_alias(
                        REGISTERED_MODEL_NAME,
                        "latest-databricks"
                    )
                    registered_version = latest_version.version
                    logger.info(f"Versión registrada: {registered_version}")
                except Exception as e:
                    # Si falla (ejm. primera versión del modelo), se captura sin que sea error
                    logger.warning(f"No se pudo obtener la versión automáticamente: {e}")
                    logger.info("El modelo se registró correctamente (versión no disponible para lectura)")

        except Exception as e:
            logger.error(f"Error al registrar en MLflow: {e}")
            logger.warning("El modelo no se registró, pero se usará localmente.")
    else:
        if is_dummy:
            logger.info("Modelo dummy - no se registra en MLflow.")
        elif not REGISTERED_MODEL_NAME:
                logger.info("REGISTERED_MODEL_NAME no definido - no se registra.")
        elif not feature_names:
                logger.warning("No hay lista de features - no se puede registrar en Unity Catalog.")

    # =====================================================
    # 5. Guardar lista de features en task values
    # =====================================================
    final_feature_names = feature_names if feature_names else (FEATURE_NAMES if 'FEATURE_NAMES' in globals() else None)
    if final_feature_names:
        dbutils.jobs.taskValues.set(key="features_list", value=str(final_feature_names))
        logger.info(f"Lista de features guardada en task values ({len(final_feature_names)} columnas)")
    else:
        dbutils.jobs.taskValues.set(key="features_list", value="none")
        logger.warning("No se pudo guardar lista de features (se usará lista vacía en inferencia)")

    # =====================================
    # 6. Task values finales
    # =====================================
    duration = time.time() - start_time
    logger.log_stage_end('load_model', duration=duration, model_loaded=True, dummy=is_dummy, run_id=run_id)

    dbutils.jobs.taskValues.set(key="model_loaded", value=True)
    dbutils.jobs.taskValues.set(key="dummy_model", value=is_dummy)
    dbutils.jobs.taskValues.set(key="model_version", value=registered_version if registered_version else "none")
    if run_id:
        dbutils.jobs.taskValues.set(key="mlflow_run_id", value=run_id)

    logger.info(f"✅ Modelo cargado exitosamente. Tiempo: {duration:.2f}s")

except Exception as e:
    logger.log_exception(operation='load_model', exception=e, should_raise=True, mes_vta=MES_VTA, environment=ENV)
    dbutils.jobs.taskValues.set(key="model_loaded", value=False)

finally:
    if 'logger' in locals():
        logger.info(f"Finalizando notebook: {logger.name}")
        logger._flush_all_handlers()
        logger.close()